## Multirun Stability Analysis: Learning Rate, Dtype, and Normalization
This notebook analyzes runs created with `train_multirun.sbatch`-style naming.
It compares:
- learning rate (`probe_head_lr`) effects
- dtype/normalization variant effects (`ln_fp32` vs `fp32_only`)
- Apertus-trained vs Llama-trained probes on matching layers.

In [ ]:
import re
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import wandb

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

api = wandb.Api()
ENTITY = "ethz-lsai-25"
PROJECT = "hallucination-probes"
RUN_PATH = f"{ENTITY}/{PROJECT}"

: 

In [ ]:
TRAIN_MODELS = ["apertus", "llama"]
LAYERS = [10, 20, 26, 30]
SEEDS = [42, 43, 44]
LRS = ["3e-4", "1e-4", "3e-5"]
VARIANTS = ["ln_fp32", "fp32_only"]
TEST_MODELS = ["apertus", "llama"]
METRIC_FAMILIES = ["all", "span", "span_max"]
METRICS = ["auc", "f1", "acc"]

LR_TAG_TO_VALUE = {"3em4": "3e-4", "1em4": "1e-4", "3em5": "3e-5"}
RUN_NAME_PATTERN = re.compile(
    r"^(apertus|llama)_no_lora_(ln_fp32|fp32_only)_layer(10|20|26|30)_seed(42|43|44)_lr(3em4|1em4|3em5)$"
)

def _extract_probe_id(run) -> Optional[str]:
    name = run.name or ""
    if RUN_NAME_PATTERN.match(name):
        return name

    cfg = run.config or {}

    if isinstance(cfg.get("probe_config"), dict):
        nested = cfg["probe_config"].get("probe_id")
        if isinstance(nested, str) and RUN_NAME_PATTERN.match(nested):
            return nested

    for key in ["probe_config.probe_id", "probe_config/probe_id", "probe_id"]:
        value = cfg.get(key)
        if isinstance(value, str) and RUN_NAME_PATTERN.match(value):
            return value

    return None

def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

In [ ]:
rows: List[Dict] = []
for run in api.runs(RUN_PATH):
    probe_id = _extract_probe_id(run)
    if probe_id is None:
        continue

    m = RUN_NAME_PATTERN.match(probe_id)
    if m is None:
        continue

    train_model, variant, layer, seed, lr_tag = m.groups()
    summary = run.summary._json_dict

    row = {
        "run_id": run.id,
        "run_name": run.name,
        "probe_id": probe_id,
        "state": run.state,
        "created_at": pd.to_datetime(run.created_at, utc=True),
        "train_model": train_model,
        "variant": variant,
        "layer": int(layer),
        "seed": int(seed),
        "lr": LR_TAG_TO_VALUE[lr_tag],
        "probe_dtype": "float32",
        "normalize_before_head": "layernorm" if variant == "ln_fp32" else "none",
    }

    for test_model in TEST_MODELS:
        for family in METRIC_FAMILIES:
            for metric in METRICS:
                key = f"train/longfact_test_{test_model}/{family}_{metric}"
                row[key] = _safe_float(summary.get(key))

    rows.append(row)

df_runs = pd.DataFrame(rows)
if df_runs.empty:
    raise ValueError("No multirun-style runs found.")

state_rank = {"finished": 0, "running": 1, "queued": 2, "failed": 3, "crashed": 4}
df_runs["_state_rank"] = df_runs["state"].map(state_rank).fillna(99)
df_runs = (
    df_runs.sort_values(["_state_rank", "created_at"], ascending=[True, False])
    .drop_duplicates(["train_model", "variant", "layer", "seed", "lr"], keep="first")
    .drop(columns=["_state_rank"])
    .sort_values(["train_model", "variant", "layer", "seed", "lr"])
    .reset_index(drop=True)
)

print(f"Collected {len(df_runs)} unique runs after de-duplication.")
df_runs.head()

In [ ]:
expected = len(TRAIN_MODELS) * len(VARIANTS) * len(LAYERS) * len(SEEDS) * len(LRS)
print(f"Expected (if both train models are present): {expected}")
print(f"Available: {len(df_runs)}")

coverage = (
    df_runs.groupby(["train_model", "variant", "layer", "lr"]).size()
    .rename("num_seeds")
    .reset_index()
)
coverage.head(20)

In [ ]:
metric_columns = [
    f"train/longfact_test_{test_model}/{family}_{metric}"
    for test_model in TEST_MODELS
    for family in METRIC_FAMILIES
    for metric in METRICS
]

df_long = df_runs.melt(
    id_vars=[
        "run_id", "probe_id", "state", "created_at", "train_model", "variant",
        "probe_dtype", "normalize_before_head", "layer", "seed", "lr"
    ],
    value_vars=metric_columns,
    var_name="metric_key",
    value_name="value",
)

parsed = df_long["metric_key"].str.extract(
    r"train/longfact_test_(apertus|llama)/(all|span|span_max)_(auc|f1|acc)"
)
parsed.columns = ["test_model", "metric_family", "metric"]

df_long = pd.concat([df_long, parsed], axis=1)
df_long = df_long.dropna(subset=["value", "test_model", "metric_family", "metric"]).copy()
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"]).reset_index(drop=True)

df_long["lr"] = pd.Categorical(df_long["lr"], categories=LRS, ordered=True)
df_long["variant"] = pd.Categorical(df_long["variant"], categories=VARIANTS, ordered=True)

df_long.head()

In [ ]:
def plot_lr_effect(df, metric_family="all", metric="auc", test_model="apertus"):
    sub = df[(df["metric_family"] == metric_family) & (df["metric"] == metric) & (df["test_model"] == test_model)].copy()
    if sub.empty:
        raise ValueError("No rows for the selected filters.")

    agg = (
        sub.groupby(["train_model", "variant", "layer", "lr"])
        .agg(mean=("value", "mean"), std=("value", "std"), n=("value", "size"))
        .reset_index()
    )
    agg["std"] = agg["std"].fillna(0.0)

    g = sns.FacetGrid(
        agg,
        row="variant",
        col="layer",
        hue="train_model",
        margin_titles=True,
        sharey=False,
        height=3.2,
        aspect=1.0,
    )
    g.map_dataframe(sns.lineplot, x="lr", y="mean", marker="o", linewidth=2.0)

    for ax, (_, panel_df) in zip(g.axes.flat, agg.groupby(["variant", "layer"], sort=False)):
        for tm in panel_df["train_model"].unique():
            d = panel_df[panel_df["train_model"] == tm].sort_values("lr")
            ax.fill_between(d["lr"], d["mean"] - d["std"], d["mean"] + d["std"], alpha=0.15)
        ax.tick_params(axis="x", rotation=30)

    g.add_legend(title="Probe trained on")
    g.set_axis_labels("Learning rate", f"{metric.upper()}")
    g.fig.suptitle(
        f"LR effect by layer and variant | test={test_model}, family={metric_family}, metric={metric}",
        y=1.03,
        fontsize=12,
    )
    plt.show()


plot_lr_effect(df_long, metric_family="all", metric="auc", test_model="apertus")
plot_lr_effect(df_long, metric_family="all", metric="f1", test_model="apertus")

In [ ]:
def summarize_best_configs(df, metric_family="all", metric="auc"):
    sub = df[(df["metric_family"] == metric_family) & (df["metric"] == metric)].copy()
    agg = (
        sub.groupby(["test_model", "train_model", "layer", "variant", "probe_dtype", "normalize_before_head", "lr"])
        .agg(mean=("value", "mean"), std=("value", "std"), n=("value", "size"))
        .reset_index()
    )
    agg["std"] = agg["std"].fillna(0.0)

    winners = (
        agg.sort_values(["test_model", "train_model", "layer", "mean"], ascending=[True, True, True, False])
        .drop_duplicates(["test_model", "train_model", "layer"], keep="first")
        .sort_values(["test_model", "train_model", "layer"])
        .reset_index(drop=True)
    )
    return winners

best_auc = summarize_best_configs(df_long, metric_family="all", metric="auc")
best_f1 = summarize_best_configs(df_long, metric_family="all", metric="f1")

print("Best AUC config per (test_model, train_model, layer):")
display(best_auc)
print("Best F1 config per (test_model, train_model, layer):")
display(best_f1)